In [0]:
# 1. CẤU HÌNH VÀ TẠO SCHEMA GOLD TRONG CATALOG
CATALOG_NAME = "omni_retail_catalog"
SCHEMA_NAME = "gold"

spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")

print(f"Đang làm việc tại: {CATALOG_NAME}.{SCHEMA_NAME}")

# 2. XÂY DỰNG BẢNG CHIỀU: dim_product
from pyspark.sql import functions as F

silver_sales_table = f"{CATALOG_NAME}.silver.fnd_sales_items"
target_dim_prod = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_product"

df_items = spark.table(silver_sales_table)

df_dim_product = (
    df_items
    .select("product_id", "product_name", "unit_price")
    .dropDuplicates(["product_id"])
)

(
    df_dim_product.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_dim_prod)
)

print(f"Hoàn thành Gold Dim Product: {spark.table(target_dim_prod).count():,} sản phẩm.")

# 3. XÂY DỰNG BẢNG CHIỀU: dim_customer (Lọc bản ghi hiện hành is_current = True)
silver_cust_table = f"{CATALOG_NAME}.silver.dim_customer_scd2"
target_dim_cust = f"{CATALOG_NAME}.{SCHEMA_NAME}.dim_customer"

df_dim_cust = (
    spark.table(silver_cust_table)
    .filter(F.col("is_current") == True)
    .select("customer_id", "customer_name", "email", "city", "state")
)

(
    df_dim_cust.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_dim_cust)
)

print(f"Hoàn thành Gold Dim Customer: {spark.table(target_dim_cust).count():,} khách hàng.")

# 4. XÂY DỰNG BẢNG SỰ KIỆN: fact_sales (Transaction Fact Table chuẩn Star Schema)
# Giữ order_id làm Degenerate Dimension để hỗ trợ DISTINCTCOUNT chính xác trên Power BI
target_fact_sales = f"{CATALOG_NAME}.{SCHEMA_NAME}.fact_sales_daily"

df_fact_sales = (
    df_items
    .select(
        F.col("order_id"),
        F.col("order_date"),
        F.col("customer_id"),
        F.col("product_id"),
        F.col("quantity"),
        F.col("unit_price"),
        F.col("line_total").alias("total_sales_amount")
    )
)

(
    df_fact_sales.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_fact_sales)
)

print(f"Hoàn thành Gold Fact Sales: {spark.table(target_fact_sales).count():,} dòng dữ liệu giao dịch.")

# 5. BẢO TRÌ & TỐI ƯU HIỆU NĂNG BẢNG GOLD DELTA (Compaction & File Sizing)
spark.sql(f"OPTIMIZE {target_fact_sales}")
print("Đã hoàn tất tối ưu bảng Gold Fact Sales.")